# 01 · SQL: perguntas de negócio
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/previsao-faltas-consultas/blob/main/notebooks/01_sql_perguntas_negocio.ipynb)

Modelo: `pacientes` 1→N `consultas` N→1 `bairros`. Cada pergunta termina com **o que aprendi**.

In [1]:
# Funções de apoio do projeto (mesma carga, mesmas variáveis e mesmo modelo em todos os notebooks)
import os, sys, urllib.request
try:
    import google.colab
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jessicapmartins/previsao-faltas-consultas/main/src/apoio.py', '/content/apoio.py')
    sys.path.insert(0, '/content')
except ModuleNotFoundError:
    sys.path.insert(0, '../src')
import apoio, pandas as pd, numpy as np

con, DADOS, CSV = apoio.conectar()

def q(sql):
    """Roda uma consulta SQL no banco de consultas e devolve um DataFrame."""
    return pd.read_sql(sql, con)

apoio.garantir(con, CSV)

## 0. Conhecer as tabelas e colunas

In [2]:
for t in ['consultas', 'pacientes', 'bairros']:
    print(f'--- {t}'); display(q(f'SELECT * FROM {t} LIMIT 3'))

--- consultas


,consulta_id,paciente_id,bairro_id,data_agendamento,data_consulta,idade,sms_recebido,faltou
0,5642903,29872499824296,39,2016-04-29 18:38:08,2016-04-29,62,0,0
1,5642503,558997776694438,39,2016-04-29 16:08:27,2016-04-29,56,0,0
2,5642549,4262962299951,45,2016-04-29 16:19:04,2016-04-29,62,0,0


--- pacientes


,paciente_id,genero,bolsa_familia,hipertensao,diabetes,alcoolismo,deficiencia
0,29872499824296,F,0,1,0,0,0
1,558997776694438,M,0,0,0,0,0
2,4262962299951,F,0,0,0,0,0


--- bairros


,bairro_id,bairro
0,0,AEROPORTO
1,1,ANDORINHAS
2,2,ANTÔNIO HONÓRIO


## 1. Taxa geral de faltas

In [3]:
q('''
SELECT COUNT(*)                     AS consultas,
       SUM(faltou)                  AS faltas,
       ROUND(100.0 * AVG(faltou),1) AS taxa_falta_pct,
       MIN(data_consulta)           AS primeira_consulta,
       MAX(data_consulta)           AS ultima_consulta
FROM consultas
''')

,consultas,faltas,taxa_falta_pct,primeira_consulta,ultima_consulta
0,110527,22319,20.2,2016-04-29,2016-06-08


**O que aprendi:** são 110.527 consultas entre 29/04 e 08/06/2016 (cerca de 6 semanas), com 22.319 faltas: **20,2%**. Um em cada cinco horários fica vago.

## 2. O SMS reduz faltas?
Primeiro a comparação simples. Depois a mesma comparação **dentro de faixas de antecedência**, porque o SMS pode ser enviado só para quem agendou com antecedência.

In [4]:
q('''
SELECT sms_recebido,
       COUNT(*)                      AS consultas,
       ROUND(100.0 * AVG(faltou), 1) AS taxa_falta_pct
FROM consultas
GROUP BY sms_recebido
''')

,sms_recebido,consultas,taxa_falta_pct
0,0,75045,16.7
1,1,35482,27.6


In [5]:
q('''
WITH c AS (
    SELECT sms_recebido, faltou,
           CAST(julianday(data_consulta) - julianday(date(data_agendamento)) AS INTEGER) AS dias
    FROM consultas
)
SELECT CASE WHEN dias = 0  THEN '0 · mesmo dia'
            WHEN dias <= 2 THEN '1 · 1 a 2 dias'
            WHEN dias <= 7 THEN '2 · 3 a 7 dias'
            WHEN dias <= 30 THEN '3 · 8 a 30 dias'
            ELSE '4 · mais de 30 dias' END                         AS antecedencia,
       SUM(CASE WHEN sms_recebido = 1 THEN 1 ELSE 0 END)           AS consultas_com_sms,
       SUM(CASE WHEN sms_recebido = 0 THEN 1 ELSE 0 END)           AS consultas_sem_sms,
       ROUND(100.0 * AVG(CASE WHEN sms_recebido = 1 THEN faltou END), 1) AS falta_com_sms_pct,
       ROUND(100.0 * AVG(CASE WHEN sms_recebido = 0 THEN faltou END), 1) AS falta_sem_sms_pct
FROM c
WHERE dias >= 0
GROUP BY antecedencia
ORDER BY antecedencia
''')

,antecedencia,consultas_com_sms,consultas_sem_sms,falta_com_sms_pct,falta_sem_sms_pct
0,0 · mesmo dia,0,38563,NaN,4.6
1,1 · 1 a 2 dias,0,11938,NaN,22.7
2,2 · 3 a 7 dias,11548,8699,23.8,26.6
3,3 · 8 a 30 dias,17524,11872,29.1,35.6
4,4 · mais de 30 dias,6410,3968,30.2,37.5


**O que aprendi:** na comparação simples, quem recebeu SMS falta **mais** (27,6% contra 16,7%). Mas nenhuma consulta marcada com até 2 dias de antecedência recebeu SMS, e essas são as que menos faltam. Dentro de cada faixa de antecedência, quem recebeu SMS falta **menos**: 23,8% contra 26,6% (3 a 7 dias), 29,1% contra 35,6% (8 a 30) e 30,2% contra 37,5% (mais de 30). É um exemplo de **paradoxo de Simpson**. Mesmo assim, isso não prova que o SMS reduz faltas: não sabemos o critério de envio (por exemplo, ter celular cadastrado), e esse critério pode estar ligado à falta.

## 3. Antecedência do agendamento
Taxa de falta por faixa de dias entre agendamento e consulta.

*Conceitos:* `julianday()`, `CASE`.

In [6]:
q('''
WITH c AS (
    SELECT faltou, CAST(julianday(data_consulta) - julianday(date(data_agendamento)) AS INTEGER) AS dias
    FROM consultas
)
SELECT CASE WHEN dias < 0   THEN '(erro) consulta antes do agendamento'
            WHEN dias = 0   THEN '0 · mesmo dia'
            WHEN dias <= 7  THEN '1 · 1 a 7 dias'
            WHEN dias <= 30 THEN '2 · 8 a 30 dias'
            ELSE                 '3 · mais de 30 dias' END AS antecedencia,
       COUNT(*)                      AS consultas,
       ROUND(100.0 * AVG(faltou), 1) AS taxa_falta_pct
FROM c
GROUP BY antecedencia
ORDER BY antecedencia
''')

,antecedencia,consultas,taxa_falta_pct
0,(erro) consulta antes do agendamento,5,100.0
1,0 · mesmo dia,38563,4.6
2,1 · 1 a 7 dias,32185,24.1
3,2 · 8 a 30 dias,29396,31.7
4,3 · mais de 30 dias,10378,33.0


**O que aprendi:** a antecedência é o fator mais forte. Consultas do **mesmo dia** (38.563, 35% da base) têm só 4,6% de falta; de 1 a 7 dias, 24,1%; de 8 a 30 dias, 31,7%; mais de 30 dias, 33,0%. Cinco consultas aparecem com data anterior ao agendamento: erro de cadastro.

## 4. Dia da semana da consulta
*Conceitos:* `strftime('%w', data_consulta)` (0 = domingo).

In [7]:
q('''
SELECT CAST(strftime('%w', data_consulta) AS INTEGER) AS dia,
       CASE strftime('%w', data_consulta)
            WHEN '1' THEN 'segunda' WHEN '2' THEN 'terça' WHEN '3' THEN 'quarta'
            WHEN '4' THEN 'quinta'  WHEN '5' THEN 'sexta' WHEN '6' THEN 'sábado' END AS dia_semana,
       COUNT(*)                      AS consultas,
       ROUND(100.0 * AVG(faltou), 1) AS taxa_falta_pct
FROM consultas
GROUP BY dia
ORDER BY dia
''')

,dia,dia_semana,consultas,taxa_falta_pct
0,1,segunda,22715,20.6
1,2,terça,25640,20.1
2,3,quarta,25867,19.7
3,4,quinta,17247,19.4
4,5,sexta,19019,21.2
5,6,sábado,39,23.1


**O que aprendi:** o dia da semana quase não muda a taxa: vai de 19,4% (quinta) a 21,2% (sexta). Sábado tem só 39 consultas. Não é uma alavanca de gestão.

## 5. Faixa etária e gênero
*Conceitos:* JOIN com `pacientes`, `CASE` para faixas de idade.

In [8]:
q('''
SELECT CASE WHEN c.idade < 0  THEN '(erro) idade negativa'
            WHEN c.idade < 13 THEN '0 a 12'
            WHEN c.idade < 19 THEN '13 a 18'
            WHEN c.idade < 30 THEN '19 a 29'
            WHEN c.idade < 45 THEN '30 a 44'
            WHEN c.idade < 60 THEN '45 a 59'
            ELSE                   '60 ou mais' END AS faixa_etaria,
       COUNT(*)                                                         AS consultas,
       ROUND(100.0 * AVG(CASE WHEN p.genero = 'F' THEN c.faltou END), 1) AS falta_mulheres_pct,
       ROUND(100.0 * AVG(CASE WHEN p.genero = 'M' THEN c.faltou END), 1) AS falta_homens_pct,
       ROUND(100.0 * AVG(c.faltou), 1)                                  AS falta_total_pct
FROM consultas c
JOIN pacientes p ON p.paciente_id = c.paciente_id
GROUP BY faixa_etaria
ORDER BY MIN(c.idade)
''')

,faixa_etaria,consultas,falta_mulheres_pct,falta_homens_pct,falta_total_pct
0,(erro) idade negativa,1,0.0,NaN,0.0
1,0 a 12,21036,20.5,20.5,20.5
2,13 a 18,7830,25.5,26.9,26.1
3,19 a 29,15244,24.9,24.3,24.8
4,30 a 44,22022,21.9,21.7,21.8
5,45 a 59,23221,18.0,17.7,17.9
6,60 ou mais,21173,15.7,14.6,15.3


**O que aprendi:** a falta cai com a idade: 26,1% entre 13 e 18 anos e 24,8% entre 19 e 29, contra 15,3% a partir de 60. Mulheres (20,3%) e homens (20,0%) faltam igual.

## 6. Bairros com maior taxa de falta
Só bairros com pelo menos 500 consultas. Ranking com `RANK()`.

*Conceitos:* JOIN com `bairros`, `HAVING`, função de janela.

In [9]:
q('''
WITH por_bairro AS (
    SELECT b.bairro,
           COUNT(*)                      AS consultas,
           ROUND(100.0 * AVG(c.faltou), 1) AS taxa_falta_pct
    FROM consultas c
    JOIN bairros b ON b.bairro_id = c.bairro_id
    GROUP BY b.bairro
    HAVING COUNT(*) >= 500
)
SELECT RANK() OVER (ORDER BY taxa_falta_pct DESC) AS ranking, *
FROM por_bairro
ORDER BY ranking
LIMIT 10
''')

,ranking,bairro,consultas,taxa_falta_pct
0,1,SANTOS DUMONT,1276,28.9
1,2,SANTA CLARA,506,26.5
2,3,ITARARÉ,3514,26.3
3,4,JESUS DE NAZARETH,2853,24.4
4,5,ILHA DO PRÍNCIPE,2266,23.5
5,6,ANDORINHAS,2262,23.0
6,6,CARATOÍRA,2565,23.0
7,8,PRAIA DO SUÁ,1288,22.8
8,9,GURIGICA,2018,22.6
9,10,BENTO FERREIRA,858,22.5


In [10]:
q('''
WITH por_bairro AS (
    SELECT b.bairro, COUNT(*) AS consultas, AVG(c.faltou) AS taxa
    FROM consultas c JOIN bairros b ON b.bairro_id = c.bairro_id
    GROUP BY b.bairro HAVING COUNT(*) >= 500
)
SELECT COUNT(*)                  AS bairros_com_500_ou_mais,
       ROUND(100.0 * MIN(taxa), 1) AS menor_taxa_pct,
       ROUND(100.0 * MAX(taxa), 1) AS maior_taxa_pct
FROM por_bairro
''')

,bairros_com_500_ou_mais,menor_taxa_pct,maior_taxa_pct
0,54,15.7,28.9


**O que aprendi:** entre os 54 bairros com pelo menos 500 consultas, a taxa vai de 15,7% a 28,9% (Santos Dumont). Itararé junta volume alto (3.514 consultas) e taxa alta (26,3%). O bairro carrega informação, mas provavelmente reflete outros fatores, como renda e distância até a unidade.

## 7. Condições de saúde e bolsa família
Taxa de falta de quem tem cada condição, comparada a quem não tem.

*Conceitos:* `UNION ALL` para empilhar uma consulta por condição.

In [11]:
q('''
WITH b AS (SELECT c.faltou, p.* FROM consultas c JOIN pacientes p ON p.paciente_id = c.paciente_id)
SELECT 'Bolsa família' AS grupo,
       ROUND(100.0 * AVG(CASE WHEN bolsa_familia = 1 THEN faltou END), 1) AS com_pct,
       ROUND(100.0 * AVG(CASE WHEN bolsa_familia = 0 THEN faltou END), 1) AS sem_pct FROM b
UNION ALL
SELECT 'Hipertensão',
       ROUND(100.0 * AVG(CASE WHEN hipertensao = 1 THEN faltou END), 1),
       ROUND(100.0 * AVG(CASE WHEN hipertensao = 0 THEN faltou END), 1) FROM b
UNION ALL
SELECT 'Diabetes',
       ROUND(100.0 * AVG(CASE WHEN diabetes = 1 THEN faltou END), 1),
       ROUND(100.0 * AVG(CASE WHEN diabetes = 0 THEN faltou END), 1) FROM b
UNION ALL
SELECT 'Alcoolismo',
       ROUND(100.0 * AVG(CASE WHEN alcoolismo = 1 THEN faltou END), 1),
       ROUND(100.0 * AVG(CASE WHEN alcoolismo = 0 THEN faltou END), 1) FROM b
UNION ALL
SELECT 'Deficiência',
       ROUND(100.0 * AVG(CASE WHEN deficiencia > 0 THEN faltou END), 1),
       ROUND(100.0 * AVG(CASE WHEN deficiencia = 0 THEN faltou END), 1) FROM b
''')

,grupo,com_pct,sem_pct
0,Bolsa família,23.7,19.8
1,Hipertensão,17.3,20.9
2,Diabetes,18.0,20.4
3,Alcoolismo,20.1,20.2
4,Deficiência,18.2,20.2


**O que aprendi:** quem tem bolsa família falta mais (23,7% contra 19,8%). Quem tem hipertensão (17,3%) ou diabetes (18,0%) falta menos; uma hipótese é o acompanhamento contínuo dessas condições, mas a base não permite confirmar. Alcoolismo e deficiência quase não mudam a taxa.

## 8. Histórico do paciente *(vai virar variável do modelo)*
Para cada consulta, quantas faltas o paciente teve **em dias anteriores**. Depois, a taxa de falta conforme esse histórico.

*Conceitos:* `SUM(faltou) OVER (PARTITION BY paciente_id ORDER BY ... RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)`. Usamos `RANGE` com a data em número (`julianday`) para olhar só **dias anteriores**: consultas do mesmo dia não entram, porque o resultado delas ainda não é conhecido.

⚠️ Só use informação anterior à consulta, senão o modelo "vê o futuro" (vazamento).

In [12]:
q('''
WITH hist AS (
    SELECT faltou,
           COUNT(*) OVER w             AS consultas_anteriores,
           COALESCE(SUM(faltou) OVER w, 0) AS faltas_anteriores
    FROM consultas
    WINDOW w AS (PARTITION BY paciente_id ORDER BY julianday(data_consulta)
                 RANGE BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)
)
SELECT CASE WHEN consultas_anteriores = 0 THEN '0 · primeira consulta na base'
            WHEN faltas_anteriores = 0    THEN '1 · já veio antes e nunca faltou'
            WHEN faltas_anteriores = 1    THEN '2 · já faltou 1 vez'
            ELSE                               '3 · já faltou 2 vezes ou mais' END AS historico,
       COUNT(*)                      AS consultas,
       ROUND(100.0 * AVG(faltou), 1) AS taxa_falta_pct
FROM hist
GROUP BY historico
ORDER BY historico
''')

,historico,consultas,taxa_falta_pct
0,0 · primeira consulta na base,66290,19.4
1,1 · já veio antes e nunca faltou,30112,18.9
2,2 · já faltou 1 vez,10483,25.1
3,3 · já faltou 2 vezes ou mais,3642,32.1


**O que aprendi:** o histórico pesa: quem já faltou 1 vez falta 25,1% na consulta seguinte, e quem já faltou 2 vezes ou mais, 32,1%, contra 18,9% de quem já veio e nunca faltou. Mas 66.290 consultas (60%) são a primeira do paciente na base, porque a janela é curta; para elas o histórico não ajuda.

## 9. Qualidade dos dados
Idades negativas ou implausíveis, consultas antes do agendamento e deficiência com valores acima de 1 (a base registra o número de deficiências).

In [13]:
q('''
SELECT 'Idade negativa' AS problema, COUNT(*) AS consultas FROM consultas WHERE idade < 0
UNION ALL
SELECT 'Idade acima de 100', COUNT(*) FROM consultas WHERE idade > 100
UNION ALL
SELECT 'Consulta antes do agendamento', COUNT(*) FROM consultas
 WHERE julianday(data_consulta) < julianday(date(data_agendamento))
UNION ALL
SELECT 'Deficiência com valor 2 a 4', COUNT(*) FROM consultas c
  JOIN pacientes p ON p.paciente_id = c.paciente_id WHERE p.deficiencia > 1
UNION ALL
SELECT 'Consulta marcada para sábado', COUNT(*) FROM consultas WHERE strftime('%w', data_consulta) = '6'
''')

,problema,consultas
0,Idade negativa,1
1,Idade acima de 100,7
2,Consulta antes do agendamento,5
3,Deficiência com valor 2 a 4,199
4,Consulta marcada para sábado,39


**O que aprendi:** os erros são poucos: 1 idade negativa, 7 idades acima de 100 e 5 consultas antes do agendamento. Essas 13 consultas saem da base de modelagem. Deficiência de 2 a 4 (199 consultas) não é erro: a base registra o número de deficiências, e o modelo usa só tem/não tem. As 39 consultas de sábado ficam.

## Conclusão
Três fatores explicam a maior parte das faltas: **antecedência do agendamento** (de 4,6% no mesmo dia a 33% acima de 30 dias), **histórico de faltas** e **idade** (jovens faltam mais). Dia da semana e gênero quase não importam. O SMS parece aumentar as faltas, mas é efeito da antecedência: dentro de cada faixa, quem recebeu SMS falta menos. Como o SMS não foi enviado ao acaso, o efeito real dele **não pode ser medido com esta base**. Esse ponto volta no notebook 05.

## Gravar no Google Drive
Fecha o banco e força o Drive a gravar agora. Sem isso, o próximo notebook pode não enxergar as tabelas novas.

In [14]:
con.commit()
con.close()
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Banco gravado no Google Drive.')
except ModuleNotFoundError:
    print('Fora do Colab: nada a sincronizar.')

Fora do Colab: nada a sincronizar.
